# Configure logging for a project

**Problem:** You have a project with multiple modules and need a consistent,
maintainable logging configuration.

This guide shows you how to set up logging configuration using dictionary-based
configuration, JSON configuration files, and best practices for multi-module projects.

## Dictionary-based configuration

The `logging.config.dictConfig()` function accepts a dictionary that describes
the entire logging configuration: formatters, handlers, and loggers. This is the
recommended approach for most projects.

In [ ]:
import logging
import logging.config
import os
import tempfile

log_dir = tempfile.mkdtemp()
log_file = os.path.join(log_dir, "app.log")

LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {
            "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        },
        "brief": {
            "format": "%(levelname)s: %(message)s"
        },
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "level": "WARNING",
            "formatter": "brief",
        },
        "file": {
            "class": "logging.FileHandler",
            "level": "DEBUG",
            "formatter": "standard",
            "filename": log_file,
        },
    },
    "loggers": {
        "my_project": {
            "level": "DEBUG",
            "handlers": ["console", "file"],
            "propagate": False,
        },
    },
    "root": {
        "level": "WARNING",
        "handlers": ["console"],
    },
}

logging.config.dictConfig(LOGGING_CONFIG)

logger = logging.getLogger("my_project")
logger.debug("Debug message (file only)")
logger.info("Info message (file only)")
logger.warning("Warning message (console and file)")

# Show file contents
for handler in logger.handlers:
    if hasattr(handler, "flush"):
        handler.flush()

print("\n--- File contents ---")
with open(log_file, "r", encoding="utf-8") as f:
    print(f.read())

### Key settings in the configuration dictionary

- **`version`**: Always set to `1` (the only valid value)
- **`disable_existing_loggers`**: Set to `False` to avoid silencing loggers created
  before `dictConfig()` is called
- **`formatters`**: Define named format strings
- **`handlers`**: Define named handlers with class, level, and formatter
- **`loggers`**: Configure specific named loggers
- **`root`**: Configure the root logger

## Using a JSON configuration file

You can store the configuration dictionary in a JSON file and load it at startup.
This separates configuration from code and makes it easy to change logging behaviour
without modifying source files.

In [ ]:
import json
import logging
import logging.config
import os
import tempfile

log_dir = tempfile.mkdtemp()
config_file = os.path.join(log_dir, "logging_config.json")
log_file = os.path.join(log_dir, "app.log")

# Create the configuration file
config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {
            "format": "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "level": "INFO",
            "formatter": "standard",
        },
        "file": {
            "class": "logging.FileHandler",
            "level": "DEBUG",
            "formatter": "standard",
            "filename": log_file,
        },
    },
    "root": {
        "level": "DEBUG",
        "handlers": ["console", "file"],
    },
}

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Configuration file created:")
with open(config_file, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Load and apply the configuration
with open(config_file, "r", encoding="utf-8") as f:
    file_config = json.load(f)

logging.config.dictConfig(file_config)

logger = logging.getLogger("json_config_demo")
logger.info("Loaded configuration from JSON file")

## Configuring multiple modules

In a multi-module project, each module should create its own logger using `__name__`.
Loggers form a hierarchy based on dot-separated names. A logger named `"my_project.database"`
is a child of the logger named `"my_project"`.

Child loggers inherit the configuration of their parent unless explicitly overridden.

In [ ]:
import logging
import logging.config

LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {
            "format": "%(name)s - %(levelname)s - %(message)s"
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "formatter": "standard",
        }
    },
    "loggers": {
        "my_project": {
            "level": "DEBUG",
            "handlers": ["console"],
            "propagate": False,
        },
        "my_project.database": {
            "level": "WARNING",
        },
    },
}

logging.config.dictConfig(LOGGING_CONFIG)

# Simulate loggers from different modules
app_logger = logging.getLogger("my_project.app")
db_logger = logging.getLogger("my_project.database")
auth_logger = logging.getLogger("my_project.auth")

app_logger.debug("App debug message -- inherits DEBUG level from parent")
db_logger.debug("Database debug -- SUPPRESSED (level is WARNING)")
db_logger.warning("Database connection slow")
auth_logger.info("User authentication successful")

## Environment-based configuration

It is common to use different logging configurations for development and production.
You can switch between them using an environment variable.

In [ ]:
import logging
import logging.config
import os


def get_logging_config(environment: str) -> dict:
    """Return a logging configuration dictionary for the given environment.

    Args:
        environment: The environment name ('development' or 'production').

    Returns:
        A logging configuration dictionary.
    """
    base_config = {
        "version": 1,
        "disable_existing_loggers": False,
        "formatters": {
            "detailed": {
                "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
            },
            "brief": {
                "format": "%(levelname)s: %(message)s"
            },
        },
        "handlers": {
            "console": {
                "class": "logging.StreamHandler",
                "formatter": "brief" if environment == "development" else "detailed",
                "level": "DEBUG" if environment == "development" else "WARNING",
            },
        },
        "root": {
            "level": "DEBUG" if environment == "development" else "INFO",
            "handlers": ["console"],
        },
    }
    return base_config


# Read environment from variable (default to 'development')
env = os.environ.get("APP_ENV", "development")
config = get_logging_config(env)
logging.config.dictConfig(config)

logger = logging.getLogger("env_demo")
logger.debug("Debug information (development only)")
logger.info("Application started")
logger.warning("Something needs attention")

print("\nCurrent environment:", env)

## Best practices

- **Configure logging once** at the application entry point (for example, in `main()` or
  `if __name__ == "__main__":`). Do not configure logging in library code.
- **Use `__name__` for logger names** so that the logger hierarchy matches the module
  hierarchy.
- **Set `disable_existing_loggers` to `False`** to avoid silencing loggers that were
  created before configuration.
- **Libraries should not configure logging.** A library should only create loggers
  and, optionally, add a `logging.NullHandler` to prevent "No handlers could be found"
  warnings.
- **Keep configuration separate from code** using JSON or YAML files when the
  configuration becomes complex.